<a href="https://colab.research.google.com/github/ReposofPriyanka/flyrank-ai-internship-ml-track/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook


## 1. Ranked actions + reason codes


### Ranked action approach

The playbook uses the transparent baseline rule developed in ML-07 to prioritize pages for human review.

Pages are flagged when they meet all three conditions:

1. Impressions are at or above the 75th-percentile volume threshold.
2. Average search position falls within positions 4–20.
3. CTR is below the impression-weighted CTR benchmark for the page's position bucket.

The baseline score estimates the number of clicks associated with the observed CTR gap:

`score = impressions × max(benchmark CTR − observed CTR, 0) / 100`

This score is used only to prioritize review. It does not predict future clicks or establish that changing a page will improve performance.

### Reason codes and actions

| Reason code | Action | Interpretation |
|---|---|---|
| `HIGH_VOLUME_LOW_CTR_FOR_POSITION` | `REVIEW_CTR` | Review the page title, search snippet, and query intent because the page has visibility but CTR below its position-band benchmark. |
| `NOT_FLAGGED` | `NO_ACTION` | The page does not meet the baseline rule. This does not mean that the page has no content or technical issues. |

Pages are ranked by score in descending order, with impressions used as a tie-breaker.

The ranked queue is a decision-support tool. A content reviewer must check page context and search intent before approving any change.

In [1]:
from google.colab import files

uploaded = files.upload()

Saving top10_review.csv to top10_review (1).csv


In [2]:
# Add human-readable action guidance to the ML-07 Top-10 review.

import pandas as pd

# Load the Top-10 review CSV uploaded to this notebook.
baseline_queue = pd.read_csv("top10_review.csv")

# Check the columns that this file actually contains.
required_columns = [
    "rank",
    "content_hash_id",
    "action",
    "reason_code",
    "score",
    "why_selected",
    "what_would_make_it_wrong"
]

missing_columns = [
    col for col in required_columns
    if col not in baseline_queue.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Add human-readable next steps.
action_guidance = {
    "HIGH_VOLUME_LOW_CTR_FOR_POSITION": (
        "Review the page title, search snippet, query intent, "
        "SERP features, and page relevance before considering a change."
    ),
    "NOT_FLAGGED": (
        "No action from this baseline rule. Other reviews may still be needed."
    )
}

baseline_queue["recommended_next_step"] = (
    baseline_queue["reason_code"]
    .map(action_guidance)
    .fillna("Manual review required: reason code is not mapped.")
)

# Add a readable priority label.
baseline_queue["review_priority"] = (
    baseline_queue["action"]
    .map({
        "REVIEW_CTR": "Review",
        "NO_ACTION": "Not selected"
    })
    .fillna("Manual review")
)

# Keep the existing ML-07 ranking.
ranked_actions = baseline_queue.sort_values("rank").copy()

preview_columns = [
    "rank",
    "content_hash_id",
    "action",
    "reason_code",
    "score",
    "why_selected",
    "recommended_next_step",
    "what_would_make_it_wrong"
]

display(ranked_actions[preview_columns])

print("Review rows:", len(ranked_actions))
print(
    "Pages flagged for CTR review:",
    int((ranked_actions["action"] == "REVIEW_CTR").sum())
)

,rank,content_hash_id,action,reason_code,score,why_selected,recommended_next_step,what_would_make_it_wrong
0,1,content_34a70fea29d15f24,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,421.641400,"143,019 impressions; position 3.2; CTR 0.0301%...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
1,2,content_7c6373141eae744a,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,347.769318,"132,593 impressions; position 5.9; CTR 0.0626%...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
2,3,content_f6116743b00afc2d,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,334.519856,"107,584 impressions; position 9.7; CTR 0.0139%...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
3,4,content_acbcc847f8996314,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,292.922550,"170,808 impressions; position 3.4; CTR 0.1534%...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
4,5,content_cd3d932d4e1c8db0,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,286.222596,"89,332 impressions; position 7.8; CTR 0.0045% ...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
5,6,content_b99ea6861864dea5,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,270.363775,"194,337 impressions; position 4.6; CTR 0.1858%...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
6,7,content_046fc480045b88f5,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,266.211200,"83,788 impressions; position 7.2; CTR 0.0072% ...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
7,8,content_f43118e089ecc69a,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,261.939190,"139,417 impressions; position 5.3; CTR 0.1370%...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
8,9,content_9540d884af3e41fd,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,256.623882,"82,376 impressions; position 8.0; CTR 0.0134% ...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...
9,10,content_425715547c6a3ea8,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,229.332071,"71,513 impressions; position 7.0; CTR 0.0042% ...","Review the page title, search snippet, query i...",The benchmark may not fit this page's query in...


Review rows: 10
Pages flagged for CTR review: 10


### Archetype-to-Action Guidance

| Page archetype | Suggested review | Human decision |
|---|---|---|
| High impressions, below-benchmark CTR | Review title, search snippet, query intent, and SERP features | Consider a title or snippet test only if the page and query context support it |
| Page with changing search intent | Compare current queries and page content | Consider whether the page needs updating to match current intent |
| Page with stable performance and no clear issue | No change based on this rule alone | Leave unchanged unless another review identifies a concern |

### Decay and Refresh Insight

A page appearing in the queue does not automatically mean its content is outdated. Reviewers should compare performance across time periods and check whether query intent, page accuracy, or search results have changed.

Consider a refresh only when the evidence suggests that the page no longer meets user needs or has outdated information. Record the reason for any refresh decision and review subsequent performance without assuming the refresh caused the change.

## 2. Intended use and limits

### Intended Use

This action queue is a human-review aid for prioritizing content pages that may have lower-than-expected click-through rates (CTR) for their search position.

The baseline rule identifies pages with high impressions and CTR below the benchmark for their position band. The queue helps reviewers decide which pages to inspect first.

### Limits

- The score is a prioritization signal, not a prediction of clicks gained.
- The CTR benchmark is descriptive and does not prove that changing a page will improve performance.
- Search performance can be affected by query intent, SERP features, seasonality, and other factors not captured by the rule.
- The queue does not determine whether a page should be rewritten, refreshed, or left unchanged.
- Recommendations require human review before any content change.
- Results may not generalize to other sites, time periods, or search contexts.

### Intended Users

Content strategists, SEO specialists, and analysts who can review page context and decide whether further investigation is appropriate.

## 3. Human review + the no-go list

### Human Review Rules

Every recommended action must be reviewed by a person before any content change is made.

Before approving an action, the reviewer should:

1. Confirm that the page and its search context are still relevant.
2. Check the page’s target queries, search intent, and current content.
3. Review the title, search snippet, SERP features, and competing results.
4. Consider whether the observed CTR difference may be explained by factors other than the content.
5. Record the decision: investigate further, make a change, or take no action.
6. Document the reason for the decision and review performance after any approved change.

### No-Go List

The baseline queue must not be used to:

- Automatically rewrite or publish page content.
- Automatically change titles, metadata, or internal links.
- Treat the score as a guaranteed estimate of clicks gained.
- Assume that low CTR proves a page has poor content.
- Make changes without checking query intent and page context.
- Delete, redirect, or significantly restructure pages based only on this queue.

### Human Accountability

The content or SEO reviewer is responsible for the final decision. The queue supports prioritization; it does not replace expert judgment.

### Cost and Value Considerations

Before approving a content change, consider:

- The potential value of improving the page for users and the business.
- The effort required to investigate and update the page.
- The risk of changing content that is already useful or performing well.
- Whether a smaller test or further investigation is appropriate.

The baseline score helps prioritize review but is not a financial value estimate or a guaranteed number of additional clicks. Reviewers should weigh the expected benefit against the effort and risk before approving work.

## 4. Monitoring / retrain triggers

### Monitoring Plan

The action queue is a prioritization aid, so monitoring should check whether its recommendations remain relevant and useful. Review the queue periodically and whenever the underlying data or business context changes.

| What to monitor | What to check | Response |
|---|---|---|
| Data quality | Missing values, duplicate pages, unexpected changes in row counts, or invalid metrics | Pause review and investigate the data issue |
| Queue stability | Whether the same pages repeatedly appear at the top of the queue | Check whether the rule is identifying persistent issues or repeatedly flagging the same pages |
| CTR benchmark | Whether the position-based CTR benchmark remains relevant for the current period | Recalculate the descriptive benchmark when the data window changes |
| Page context | Changes in query intent, SERP features, seasonality, or page purpose | Reassess the recommendation manually |
| Outcome after changes | CTR and impressions before and after an approved content change | Record the result without assuming the change caused it |

### Review and Retraining Triggers

Revisit the baseline rule or its thresholds when:

- The source data, reporting window, or metric definitions change.
- Data-quality checks identify a material issue.
- Search behaviour or SERP features change enough to affect CTR comparisons.
- The queue repeatedly flags pages that reviewers consider unsuitable for action.
- Reviewers observe that the current rule is no longer useful for prioritizing pages.

If the rule is changed, document the reason, update the notebook, and compare the new queue with the previous version.

### Monitoring Limitations

This is a lightweight review plan, not an automated production monitoring system. The queue does not learn from outcomes automatically, and changes to the rule require human review. CTR changes after a content update should be treated as observations, not proof that the update caused the change.

## 5. Exports for the paper


In [5]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue_path = output_dir / "top10_action_review.csv"

ranked_actions.to_csv(queue_path, index=False)

print("Exported Top-10 action review:")
print(queue_path)
print("Rows:", len(ranked_actions))

Exported Top-10 action review:
work/outputs/top10_action_review.csv
Rows: 10


In [6]:
from google.colab import files
files.download("work/outputs/ranked_action_queue.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.